# EE0848 MACHINE LEARNING FOR SP - WEEK10 - PRACTICE

## CNN Implementation Practice

**Objective:** Build, train, and evaluate a small Convolutional Neural Network (CNN) on CIFAR-10.

**Classroom goal:** Understand the code path from image batch to prediction: data transforms, CNN layers, loss, optimizer, training loop, and evaluation.

**Colab:** Select `Runtime > Change runtime type > GPU` before training.

## Code Map

We will go through the notebook in seven pieces:

1. Imports and device
2. CNN model
3. Image transforms
4. Dataset loading
5. Loss, optimizer, scheduler
6. Training loop
7. Evaluation

### 1. Imports and Device

The first cell loads PyTorch, torchvision, plotting tools, and selects GPU when Colab provides one.

In [ ]:
import sys
import tarfile
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### 2. Define the CNN Model

This model has two parts:

- `features`: convolution, normalization, ReLU, pooling
- `classifier`: flatten, linear layer, dropout, final logits

Shape path for CIFAR-10 images:

`3 x 32 x 32 -> 16 x 16 x 16 -> 32 x 8 x 8 -> 2048 -> 64 -> 10`

In [ ]:
class a_CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, padding=2),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),      # 32x32 -> 16x16

            nn.Conv2d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),      # 16x16 -> 8x8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

cnn_model = a_CNN(num_classes=10).to(device)
print(cnn_model)

sample_batch = torch.randn(64, 3, 32, 32).to(device)
sample_logits = cnn_model(sample_batch)
print("Input shape: ", sample_batch.shape)
print("Output shape:", sample_logits.shape)

### 3. Image Transforms

Training images get small random changes so the CNN does not memorize exact pixels. Test images are only converted and normalized.

In [ ]:
train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

### 4. Download CIFAR-10 from the Shared Course File

The notebook downloads one shared archive from Google Drive, extracts it in Colab, and then tells `torchvision` to load the local files with `download=False`.

In [ ]:
CIFAR10_FILE_ID = "1yDizhOSGiClEtba1lzJrvjL3ZMb4xI0G"

def ensure_cifar10(data_root="/content/data"):
    data_root = Path(data_root) if Path("/content").exists() else Path("./data")
    archive_path = data_root / "cifar-10-python.tar.gz"
    cifar_folder = data_root / "cifar-10-batches-py"
    data_root.mkdir(parents=True, exist_ok=True)

    if cifar_folder.exists():
        print("CIFAR-10 is already extracted.")
        return data_root

    if not archive_path.exists():
        try:
            import gdown
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
            import gdown

        url = f"https://drive.google.com/uc?id={CIFAR10_FILE_ID}"
        print("Downloading CIFAR-10 from course Google Drive...")
        gdown.download(url, str(archive_path), quiet=False)

    print("Extracting CIFAR-10...")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(path=data_root)

    return data_root

data_root = ensure_cifar10()

### 5. Build Datasets and DataLoaders

The small subset keeps the classroom run fast. Set `USE_SMALL_SUBSET = False` for full training.

In [ ]:
classes = ("plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")

train_dataset = torchvision.datasets.CIFAR10(
    root=str(data_root), train=True, download=False, transform=train_transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root=str(data_root), train=False, download=False, transform=test_transform
)

USE_SMALL_SUBSET = True
if USE_SMALL_SUBSET:
    train_dataset = Subset(train_dataset, range(5000))
    test_dataset = Subset(test_dataset, range(1000))

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples:     {len(test_dataset)}")
print(f"Training batches: {len(train_loader)}")
print(f"Test batches:     {len(test_loader)}")

### 6. Inspect One Batch

Before training, always check one batch: shape, labels, and a few images.

In [ ]:
def imshow(img):
    img = img / 2 + 0.5
    npimg = img.numpy()
    plt.figure(figsize=(6, 3))
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis("off")
    plt.show()

images, labels = next(iter(train_loader))
print("Batch image tensor:", images.shape)
print("Batch label tensor:", labels.shape)

imshow(torchvision.utils.make_grid(images[:8], nrow=4))
print("Labels:", " ".join(classes[labels[j]] for j in range(8)))

### 7. Training Setup

`CrossEntropyLoss` compares logits with class labels. `AdamW` updates the weights. The scheduler slowly changes the learning rate.

In [ ]:
cnn_model = a_CNN(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(cnn_model.parameters(), lr=1e-3, weight_decay=1e-4)

num_epochs = 2 if USE_SMALL_SUBSET else 20
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

print("Model:    ", cnn_model.__class__.__name__)
print("Loss:     ", criterion.__class__.__name__)
print("Optimizer:", optimizer.__class__.__name__)
print("Epochs:   ", num_epochs)

### 8. Training Loop

This is the core pattern repeated for each epoch:

1. Move a batch to the device
2. Run the model forward
3. Compute the loss
4. Backpropagate gradients
5. Update the weights
6. Report epoch loss and accuracy

In [ ]:
def train_model(model, train_loader, criterion, optimizer, scheduler, num_epochs, device):
    history = []

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += labels.size(0)

        if scheduler is not None:
            scheduler.step()

        epoch_loss = total_loss / total_samples
        epoch_acc = total_correct / total_samples
        history.append((epoch_loss, epoch_acc))

        print(
            f"Epoch {epoch + 1:02d}/{num_epochs} | "
            f"loss={epoch_loss:.4f} | acc={epoch_acc:.3f}"
        )

    return history

history = train_model(
    cnn_model, train_loader, criterion, optimizer, scheduler, num_epochs, device
)

### 9. Evaluation

Evaluation uses `model.eval()` and `torch.no_grad()` because we only need predictions, not gradients.

In [ ]:
def evaluate_model(model, test_loader, device):
    model.eval()
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            predictions = logits.argmax(dim=1)

            all_predictions.append(predictions.cpu())
            all_labels.append(labels.cpu())

    all_predictions = torch.cat(all_predictions)
    all_labels = torch.cat(all_labels)
    accuracy = (all_predictions == all_labels).float().mean().item()
    return accuracy, all_predictions, all_labels

test_accuracy, predictions, labels = evaluate_model(cnn_model, test_loader, device)
print(f"Test accuracy: {100 * test_accuracy:.2f}%")

### 10. Per-Class Accuracy

Overall accuracy is useful, but per-class accuracy shows which categories are harder for the model.

In [ ]:
def per_class_accuracy(predictions, labels, classes):
    for class_index, class_name in enumerate(classes):
        mask = labels == class_index
        if mask.sum() == 0:
            print(f"{class_name:5s}: no samples")
            continue

        class_acc = (predictions[mask] == labels[mask]).float().mean().item()
        print(f"{class_name:5s}: {100 * class_acc:5.1f}%")

per_class_accuracy(predictions, labels, classes)

## End of Practice Session

This notebook covered:

- CNN building blocks: convolution, batch norm, ReLU, pooling, dropout, fully-connected layers
- CIFAR-10 loading from a shared course archive
- Data augmentation and normalization
- Training with loss, optimizer, and scheduler
- Evaluation with overall and per-class accuracy